# Proyecto de Machine Learning
## Predicción del precio de seguros para mascotas

Este proyecto corresponde a la segunda fase del trabajo iniciado con el Análisis Exploratorio de Datos (EDA). Su objetivo es desarrollar un modelo de Machine Learning que permita estimar el precio de nuevas cotizaciones e identificar y cuantificar las variables con mayor influencia en su cálculo.

## Notebook 02 – Modelado

Este notebook contiene el entrenamiento y la comparación inicial de los modelos de regresión candidatos, siguiendo las decisiones documentadas en el **Documento 01 – Guía de Definición del Modelo** y en el **Documento 02 – Plan de Preprocesamiento**.

**Estado del notebook:** 🟢 Dataset preprocesado disponible (rama `feature/preprocessing` ya fusionada en `develop`). Entrenamiento en curso.


# 1. Importación de librerías

In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, cross_validate
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.ensemble import RandomForestRegressor


# 2. Configuración del entorno

In [2]:
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
np.set_printoptions(suppress=True)

RANDOM_STATE = 42


# 3. Carga del dataset preprocesado

## Objetivo

Cargar el conjunto de datos ya preprocesado (limpio y codificado), generado al ejecutar el **Notebook 01 – Preprocesamiento**.

## ⚠️ Nota

Este archivo no está en el repositorio (excluido en `.gitignore` por confidencialidad). Antes de ejecutar esta celda, cada integrante debe haber ejecutado `01_preprocessing.ipynb` una vez en su propio ordenador para generarlo en local.


In [3]:
df = pd.read_csv("../data/matriz_cotizaciones_grupos_preprocesada.csv")

print(df.shape)
df.head()


(67500, 79)


,pet_age_nr,is_pure_breed,is_dangerous,capital_vet_total,capital_vet_plus,capital_rc,precio_mensual,pet_species_1,pet_species_2,pet_gender_F,pet_gender_M,pet_sterilized_N,pet_sterilized_S,sterilized_when_No_aplica,sterilized_when_S2,sterilized_when_S3,plan_name_PET_VITAL,plan_name_PET3,grupo_raza_nombre_Abisinio,grupo_raza_nombre_Affenpinscher,grupo_raza_nombre_Airedale Terrier,grupo_raza_nombre_Akita Americano (G Perro Japones),grupo_raza_nombre_Akita Inu,grupo_raza_nombre_American Staffordshire Terrier,grupo_raza_nombre_Anglo-Francais De Petite Venerie,grupo_raza_nombre_Australian Terrier,grupo_raza_nombre_Balinés,grupo_raza_nombre_Beagle,grupo_raza_nombre_Biewer Terrier,grupo_raza_nombre_Boxer,grupo_raza_nombre_Bull Terrier,grupo_raza_nombre_Bullmastiff,grupo_raza_nombre_Caniche Enano (Poodle Mini),grupo_raza_nombre_Caniche Miniatura (Poodle Toy),grupo_raza_nombre_Exotic,grupo_raza_nombre_Gato Común/European Shorthair Cat,grupo_raza_nombre_Mestizo Gigante (Más de 45 Kg),grupo_raza_nombre_Mestizo Grande (Entre 21 y 45 Kg),grupo_raza_nombre_Mestizo Mediano (Entre 11 y 20 Kg),grupo_raza_nombre_Mestizo Miniatura (Menos de 5 Kg),grupo_raza_nombre_Mestizo Pequeño (Entre 5 y 10 Kg),grupo_raza_nombre_Shar Pei,grupo_raza_nombre_Spaniel Continental Enano De Compañía (Papillon / Phaléne),pet_race_name_Abisinio,pet_race_name_Affenpinscher,pet_race_name_Airedale Terrier,pet_race_name_Akita Americano (G Perro Japones),pet_race_name_Akita Inu,pet_race_name_American Staffordshire Terrier,pet_race_name_Anglo-Francais De Petite Venerie,pet_race_name_Australian Terrier,pet_race_name_Balinés,pet_race_name_Beagle,pet_race_name_Biewer Terrier,pet_race_name_Boxer,pet_race_name_Bull Terrier,pet_race_name_Bullmastiff,pet_race_name_Caniche Enano (Poodle Mini),pet_race_name_Caniche Miniatura (Poodle Toy),pet_race_name_Exotic,pet_race_name_Gato Común/European Shorthair Cat,pet_race_name_Mestizo Gigante (Más de 45 Kg),pet_race_name_Mestizo Grande (Entre 21 y 45 Kg),pet_race_name_Mestizo Mediano (Entre 11 y 20 Kg),pet_race_name_Mestizo Miniatura (Menos de 5 Kg),pet_race_name_Mestizo Pequeño (Entre 5 y 10 Kg),pet_race_name_Shar Pei,pet_race_name_Spaniel Continental Enano De Compañía (Papillon / Phaléne),owner_postal_code_2124,owner_postal_code_2210,owner_postal_code_28001,owner_postal_code_28220,owner_postal_code_50001,owner_municipality_Albacete,owner_municipality_Alcalá del Júcar,owner_municipality_Madrid,owner_municipality_Majadahonda,owner_municipality_Municipio no informado,owner_municipality_Zaragoza
0,6,1,0,1250.0,1000.0,0.0,19.30,1,0,0,1,1,0,1,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0
1,7,1,0,1250.0,1000.0,0.0,18.53,1,0,1,0,0,1,0,1,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0
2,7,1,0,1250.0,1000.0,0.0,19.08,1,0,1,0,0,1,0,0,1,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0
3,7,1,0,1250.0,1000.0,0.0,19.49,1,0,1,0,1,0,1,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0
4,7,1,0,1250.0,1000.0,0.0,18.86,1,0,0,1,0,1,1,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0


## División Train / Test

Se separa el dataset en entrenamiento (80%) y test (20%).

In [4]:
TARGET = "precio_mensual"

# Salvaguarda: si por error quedara alguna columna de leakage, se elimina aquí también
columnas_leakage = ["precio_anual"]
columnas_a_eliminar = [c for c in columnas_leakage if c in df.columns]
if columnas_a_eliminar:
    print("Aviso: se han eliminado columnas de leakage detectadas:", columnas_a_eliminar)
    df = df.drop(columns=columnas_a_eliminar)

X = df.drop(columns=[TARGET])
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

print("Train:", X_train.shape, "| Test:", X_test.shape)


Train: (54000, 78) | Test: (13500, 78)


# 4. Función de evaluación de modelos

## Objetivo

Función reutilizable que entrena y evalúa cada modelo mediante validación cruzada (solo sobre train), devolviendo MAE, RMSE y R².


In [5]:
def evaluar_modelo(nombre, modelo, X_train, y_train, cv=5):
    """
    Entrena y evalúa un modelo de regresión mediante validación cruzada.
    Se aplica únicamente sobre el conjunto de train.
    """
    scoring = {
        "MAE": "neg_mean_absolute_error",
        "RMSE": "neg_root_mean_squared_error",
        "R2": "r2",
    }

    resultados = cross_validate(
        modelo, X_train, y_train,
        cv=cv,
        scoring=scoring,
    )

    return {
        "Modelo": nombre,
        "MAE": -resultados["test_MAE"].mean(),
        "RMSE": -resultados["test_RMSE"].mean(),
        "R2": resultados["test_R2"].mean(),
    }


# 5. Modelo baseline

## Objetivo

Entrenar un `DummyRegressor` (predice la media) como referencia mínima.


In [6]:
baseline = DummyRegressor(strategy="mean")
resultado_baseline = evaluar_modelo("Baseline (media)", baseline, X_train, y_train)
resultado_baseline


{'Modelo': 'Baseline (media)',
 'MAE': np.float64(8.343753669427297),
 'RMSE': np.float64(10.539121213443511),
 'R2': np.float64(-7.280244837626348e-05)}

# 6. Modelos lineales de regresión

## Objetivo

Entrenar los modelos lineales acordados en el Documento 01: Regresión Lineal, Ridge, Lasso y Elastic Net, con hiperparámetros por defecto.


In [7]:
modelos_lineales = {
    "Regresión Lineal": LinearRegression(),
    "Ridge": Ridge(random_state=RANDOM_STATE),
    "Lasso": Lasso(random_state=RANDOM_STATE),
    "Elastic Net": ElasticNet(random_state=RANDOM_STATE),
}

resultados_lineales = [
    evaluar_modelo(nombre, modelo, X_train, y_train)
    for nombre, modelo in modelos_lineales.items()
]

pd.DataFrame(resultados_lineales)


,Modelo,MAE,RMSE,R2
0,Regresión Lineal,1.735341,2.321664,0.951461
1,Ridge,1.735142,2.321579,0.951464
2,Lasso,5.697438,7.585979,0.481851
3,Elastic Net,5.531146,7.330219,0.516203


# 7. Modelo de comparación: Random Forest

## Objetivo

Entrenar un Random Forest como término de comparación frente a los modelos lineales.


In [8]:
modelo_random_forest = RandomForestRegressor(random_state=RANDOM_STATE)
resultado_rf = evaluar_modelo("Random Forest", modelo_random_forest, X_train, y_train)
resultado_rf


{'Modelo': 'Random Forest',
 'MAE': np.float64(0.2730112333333333),
 'RMSE': np.float64(0.5358970899705636),
 'R2': np.float64(0.9974089665093857)}

# 8. Tabla comparativa de modelos

## Objetivo

Reunir los resultados de todos los modelos para facilitar la selección de los mejores candidatos.


In [9]:
tabla_comparativa = pd.DataFrame([
    resultado_baseline,
    *resultados_lineales,
    resultado_rf,
])

tabla_comparativa.sort_values("RMSE").reset_index(drop=True)


,Modelo,MAE,RMSE,R2
0,Random Forest,0.273011,0.535897,0.997409
1,Ridge,1.735142,2.321579,0.951464
2,Regresión Lineal,1.735341,2.321664,0.951461
3,Elastic Net,5.531146,7.330219,0.516203
4,Lasso,5.697438,7.585979,0.481851
5,Baseline (media),8.343754,10.539121,-0.000073


### Conclusiones
Random Forest obtiene el mejor rendimiento con diferencia (R²=0.997, RMSE=0.54), seguido de cerca por Ridge y Regresión Lineal (R²≈0.95). Lasso y Elastic Net obtuvieron un rendimiento notablemente inferior, probablemente por una regularización demasiado agresiva con los hiperparámetros por defecto (alpha=1.0), dado el elevado número de variables tras el encoding (79 columnas). Todos los modelos superan claramente al baseline, confirmando que las variables del dataset tienen poder predictivo real sobre el precio.

### Decisión
Se seleccionan Random Forest y Ridge para la fase de optimización de hiperparámetros: Random Forest por su rendimiento superior, y Ridge por ofrecer un resultado casi idéntico a Regresión Lineal pero con mayor robustez y mejor interpretabilidad mediante coeficientes. Se valorará también reoptimizar Lasso/Elastic Net con valores de alpha más bajos, ya que su bajo rendimiento actual puede deberse a una regularización excesiva y no a una limitación real del modelo.


# 9. Cierre

Con el dataset preprocesado ya integrado, este notebook entrena y compara el modelo baseline, los modelos lineales y Random Forest. El siguiente paso será pasar a la fase de optimización de hiperparámetros (GridSearchCV / RandomizedSearchCV con validación cruzada).
